# Покупатели: пол, возраст и регионы

Данные по покупателям каналов за 30 дней; один человек может присутствовать в обоих каналах. Малые группы не входят в разбивки, поэтому доли рассчитываются внутри доступной разбивки. Профиль покупателя не равен портрету продавца.
Покупатели Маркета и партнёра относятся к разным маршрутам покупки; различия состава не доказывают различия эффективности карточки и не исключают пересечения аудиторий.


## Как читать маршруты: методологически корректная трактовка

- **Карточка Маркета VK:** открытие карточки товара Маркета во VK, затем действие пользователя. Нажатие на кнопку может вести к чекауту внутри VK, в чат с продавцом или на другой адрес — фактический справочник назначения кнопок в выгрузке отсутствует.
- **Карточка партнёра во VK:** открытие карточки товара партнёра во VK и последующее нажатие на кнопку. Нажатие может вести к разным следующим шагам, включая оформление на площадке партнёра. Сам факт нажатия не означает автоматически оформленный или оплаченный заказ.
- **Прямой маршрут партнёра:** переход к партнёру без открытия карточки товара во VK. В данных он измеряется отдельным событием и относится к иной когорте.

Партнёрская карточка и прямой переход партнёра нельзя складывать в уникальных покупателей или конверсию. Маркет и партнёра следует сравнивать как связанные, но не одинаковые продуктовые маршруты. Для выводов нужны когортные данные по назначению кнопки, чекауту и оплате.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path.cwd().parent / 'Data'
mv = 'Маркет Вконтакте'
p1 = 'Партнёр 1'
pd.set_option('display.float_format', '{:,.2f}'.format)

gender_age = pd.read_csv(data_path / 'Market_Partner_gender_age.csv')
regions = pd.read_csv(data_path / 'Market_Partner_clients_by_regions.csv').set_index('Регион')
market = pd.read_csv(data_path / 'Market_Partner.csv').set_index('Метрика')
gender_age

,Пол,Возраст,Покупатели: Маркет Вконтакте,Заказы: Маркет Вконтакте,Покупатели: Партнёр 1,Заказы: Партнёр 1
0,Ж,35–44,66329,91318,700024,1113644
1,Ж,25–34,54635,75822,363334,574071
2,Ж,45–54,37784,53843,520765,840735
3,Ж,18–24,31697,44398,111820,168479
4,М,25–34,27331,38478,121325,169361
5,Ж,55+,21451,31908,446182,740608
6,М,35–44,20409,26903,255011,368869
7,М,18–24,16808,23143,35128,48007
8,М,45–54,7954,10851,186287,281495
9,М,55+,5105,7145,136092,213133


In [2]:
# Интерактивные визуализации
import plotly.graph_objects as go

def show_plotly(fig, title, x_title, y_title, height=500):
    fig.update_layout(
        title=dict(text=title), template='plotly_white', height=height,
        xaxis_title=x_title, yaxis_title=y_title,
        legend_title_text='Канал', margin=dict(l=70, r=30, t=80, b=90),
        font=dict(size=13), hoverlabel=dict(namelength=-1),
    )
    fig.show(config={'displaylogo': False, 'responsive': True})


In [3]:
coverage = pd.DataFrame(index=[mv, p1])
for channel in [mv, p1]:
    coverage.loc[channel, 'Покупатели по полу и возрасту'] = gender_age[f'Покупатели: {channel}'].sum()
    coverage.loc[channel, 'Покупатели по регионам'] = regions[f'Покупатели: {channel}'].sum()
    coverage.loc[channel, 'Покупатели в итогах'] = market.loc['Уникальные покупатели за 30 дней', channel]
    coverage.loc[channel, 'Заказы по полу и возрасту'] = gender_age[f'Заказы: {channel}'].sum()
    coverage.loc[channel, 'Заказы по регионам'] = regions[f'Заказы: {channel}'].sum()
    coverage.loc[channel, 'Заказы в итогах'] = market.loc['Оформленные заказы', channel]
coverage

,Покупатели по полу и возрасту,Покупатели по регионам,Покупатели в итогах,Заказы по полу и возрасту,Заказы по регионам,Заказы в итогах
Маркет Вконтакте,"295,716.00","295,850.00","296,148.00","413,147.00","413,334.00","413,740.00"
Партнёр 1,"2,897,475.00","2,901,162.00","2,901,334.00","4,551,464.00","4,557,253.00","4,927,525.00"


## Пол и возраст

Индекс аффинитивности = доля сегмента среди покупателей Маркета / доля сегмента среди покупателей партнёра × 100. Индекс 100 означает равенство долей, но не равенство численности.

In [4]:
profiles = gender_age.copy()
for channel in [mv, p1]:
    profiles[f'Доля покупателей, %: {channel}'] = profiles[f'Покупатели: {channel}'] / profiles[f'Покупатели: {channel}'].sum() * 100
    profiles[f'Заказов на покупателя: {channel}'] = profiles[f'Заказы: {channel}'] / profiles[f'Покупатели: {channel}'].replace(0, np.nan)
profiles['Индекс аффинитивности'] = profiles[f'Доля покупателей, %: {mv}'] / profiles[f'Доля покупателей, %: {p1}'].replace(0, np.nan) * 100
profiles.sort_values('Индекс аффинитивности', ascending=False)

,Пол,Возраст,Покупатели: Маркет Вконтакте,Заказы: Маркет Вконтакте,Покупатели: Партнёр 1,Заказы: Партнёр 1,"Доля покупателей, %: Маркет Вконтакте",Заказов на покупателя: Маркет Вконтакте,"Доля покупателей, %: Партнёр 1",Заказов на покупателя: Партнёр 1,Индекс аффинитивности
7,М,18–24,16808,23143,35128,48007,5.68,1.38,1.21,1.37,468.82
11,М,до 18,2847,4695,6020,8876,0.96,1.65,0.21,1.47,463.38
3,Ж,18–24,31697,44398,111820,168479,10.72,1.40,3.86,1.51,277.74
4,М,25–34,27331,38478,121325,169361,9.24,1.41,4.19,1.40,220.72
10,Ж,до 18,3366,4643,15487,24186,1.14,1.38,0.53,1.56,212.96
1,Ж,25–34,54635,75822,363334,574071,18.48,1.39,12.54,1.58,147.34
0,Ж,35–44,66329,91318,700024,1113644,22.43,1.38,24.16,1.59,92.84
6,М,35–44,20409,26903,255011,368869,6.90,1.32,8.80,1.45,78.42
2,Ж,45–54,37784,53843,520765,840735,12.78,1.43,17.97,1.61,71.09
5,Ж,55+,21451,31908,446182,740608,7.25,1.49,15.40,1.66,47.11


In [5]:
gender = gender_age.groupby('Пол')[[f'Покупатели: {mv}', f'Покупатели: {p1}']].sum()
age = gender_age.groupby('Возраст')[[f'Покупатели: {mv}', f'Покупатели: {p1}']].sum()
for channel in [mv, p1]:
    gender[f'Доля, %: {channel}'] = gender[f'Покупатели: {channel}'] / gender[f'Покупатели: {channel}'].sum() * 100
    age[f'Доля, %: {channel}'] = age[f'Покупатели: {channel}'] / age[f'Покупатели: {channel}'].sum() * 100
pd.concat({'Пол': gender, 'Возраст': age})

Покупатели: Маркет Вконтакте  Покупатели: Партнёр 1  \
Пол     Ж                            215262                2157612   
        М                             80454                 739863   
Возраст 18–24                         48505                 146948   
        25–34                         81966                 484659   
        35–44                         86738                 955035   
        45–54                         45738                 707052   
        55+                           26556                 582274   
        до 18                          6213                  21507   

               Доля, %: Маркет Вконтакте  Доля, %: Партнёр 1  
Пол     Ж                          72.79               74.47  
        М                          27.21               25.53  
Возраст 18–24                      16.40                5.07  
        25–34                      27.72               16.73  
        35–44                      29.33               32.96  
        45–54                      15.47               24.40  
        55+                         8.98               20.10  
        до 18                       2.10                0.74

### Интерактивно: возрастная структура покупателей
Доли рассчитаны отдельно внутри доступной возрастной разбивки каждого канала, а не среди всех покупателей в итоговой таблице.


In [6]:
age_order = ['до 18', '18–24', '25–34', '35–44', '45–54', '55+']
# Сохраняем все исходные группы, даже если их обозначения изменятся.
age_order = [x for x in age_order if x in age.index] + [x for x in age.index if x not in age_order]
fig = go.Figure()
for channel, color in [(mv, '#3978C6'), (p1, '#F2994A')]:
    fig.add_bar(name=channel, x=age_order, y=age.loc[age_order, f'Доля, %: {channel}'],
                customdata=age.loc[age_order, f'Покупатели: {channel}'], marker_color=color,
                hovertemplate='%{x}<br>%{fullData.name}: %{y:.2f}%<br>Покупатели: %{customdata:,.0f}<extra></extra>')
fig.update_layout(barmode='group')
fig.update_yaxes(rangemode='tozero', ticksuffix='%')
show_plotly(fig, 'Возраст покупателей в доступной разбивке', 'Возрастная группа', 'Доля покупателей канала, %', 470)


### Интерактивно: структура покупателей по полу
Доли отражают только категории пола, присутствующие в исходной разбивке; данные не доказывают причин различий.


In [7]:
fig = go.Figure()
for channel, color in [(mv, '#3978C6'), (p1, '#F2994A')]:
    fig.add_bar(name=channel, x=gender.index, y=gender[f'Доля, %: {channel}'],
                customdata=gender[f'Покупатели: {channel}'], marker_color=color,
                hovertemplate='%{x}<br>%{fullData.name}: %{y:.2f}%<br>Покупатели: %{customdata:,.0f}<extra></extra>')
fig.update_layout(barmode='group')
fig.update_yaxes(range=[0, 100], ticksuffix='%')
show_plotly(fig, 'Пол покупателей в доступной разбивке', 'Пол в исходных данных', 'Доля покупателей канала, %', 430)


Женщины составляют около 73% покупателей Маркета и 75% партнёра в доступной разбивке. Доля возраста 18–34 у Маркета около 44%, у партнёра — 22%; 45+ — около 25% и 45% соответственно. Это различие профилей, не доказательство того, что выбор канала вызван возрастом. Заказы на покупателя внутри каждого сегмента представлены в таблице.

## Регионы и пропуски

Четыре пропуска Маркета сохраняются как неизвестные. Строка «Не указан» исключается из географических долей и межрегиональных сопоставлений, но показывается отдельно. У партнёра она составляет большую часть базы, поэтому сравнение географических структур может быть смещено.

In [8]:
regions[regions.isna().any(axis=1)]

,Покупатели: Маркет Вконтакте,Заказы: Маркет Вконтакте,Покупатели: Партнёр 1,Заказы: Партнёр 1
Регион,,,,
Еврейская автономная область,NaN,NaN,592,952
Карачаево-Черкесская Республика,NaN,NaN,489,793
Херсонская область,NaN,NaN,258,388
Чукотский автономный округ,NaN,NaN,213,365


In [9]:
regions.loc[regions.index.str.contains('Не указан', case=False)]

,Покупатели: Маркет Вконтакте,Заказы: Маркет Вконтакте,Покупатели: Партнёр 1,Заказы: Партнёр 1
Регион,,,,
Не указан,"5,189.00","7,446.00",1065094,1677589


In [10]:
geo = regions.loc[~regions.index.str.contains('Не указан', case=False)].copy()
for channel in [mv, p1]:
    geo[f'Доля известных регионов, %: {channel}'] = geo[f'Покупатели: {channel}'] / geo[f'Покупатели: {channel}'].sum() * 100
    geo[f'Заказов на покупателя: {channel}'] = geo[f'Заказы: {channel}'] / geo[f'Покупатели: {channel}'].replace(0, np.nan)
geo['Доля Маркета в сумме покупателей каналов, %'] = geo[f'Покупатели: {mv}'] / (geo[f'Покупатели: {mv}'] + geo[f'Покупатели: {p1}']).replace(0, np.nan) * 100
geo.sort_values(f'Покупатели: {mv}', ascending=False).head(15)

,Покупатели: Маркет Вконтакте,Заказы: Маркет Вконтакте,Покупатели: Партнёр 1,Заказы: Партнёр 1,"Доля известных регионов, %: Маркет Вконтакте",Заказов на покупателя: Маркет Вконтакте,"Доля известных регионов, %: Партнёр 1",Заказов на покупателя: Партнёр 1,"Доля Маркета в сумме покупателей каналов, %"
Регион,,,,,,,,,
г. Москва,"30,488.00","42,652.00",146366,229256,10.49,1.40,7.97,1.57,17.24
г. Санкт-Петербург,"24,687.00","36,107.00",155546,248956,8.49,1.46,8.47,1.60,13.70
Челябинская область,"20,465.00","28,558.00",67879,107682,7.04,1.40,3.70,1.59,23.17
Пермский край,"13,366.00","19,489.00",54523,85703,4.60,1.46,2.97,1.57,19.69
Свердловская область,"11,117.00","15,626.00",61132,93602,3.82,1.41,3.33,1.53,15.39
Нижегородская область,"10,136.00","13,948.00",58993,91907,3.49,1.38,3.21,1.56,14.66
Самарская область,"9,627.00","13,214.00",50590,77845,3.31,1.37,2.76,1.54,15.99
Удмуртская Республика,"7,458.00","9,862.00",31199,49233,2.57,1.32,1.70,1.58,19.29
Республика Башкортостан,"6,205.00","8,330.00",42619,64042,2.13,1.34,2.32,1.50,12.71


### Интерактивно: крупнейшие известные регионы Маркета
Показаны 10 регионов с наибольшим известным числом покупателей Маркета. «Не указан» и строки с пропущенным значением исключены из ранжирования, а не приравнены к нулю.


In [11]:
region_top = geo[f'Покупатели: {mv}'].dropna().nlargest(10).sort_values()
fig = go.Figure(go.Bar(x=region_top.values, y=region_top.index, orientation='h', marker_color='#3978C6',
                       hovertemplate='%{y}<br>Покупатели Маркета: %{x:,.0f}<extra></extra>'))
fig.update_xaxes(rangemode='tozero')
show_plotly(fig, 'Топ-10 известных регионов Маркета', 'Покупатели за 30 дней, чел.', '', 600)


In [12]:
concentration = pd.Series({f'Топ-10, доля известных покупателей, %: {channel}': geo[f'Покупатели: {channel}'].nlargest(10).sum() / geo[f'Покупатели: {channel}'].sum() * 100 for channel in [mv, p1]})
concentration

Топ-10, доля известных покупателей, %: Маркет Вконтакте   48.02
Топ-10, доля известных покупателей, %: Партнёр 1          40.45
dtype: float64

In [13]:
geo[[f'Покупатели: {mv}', f'Покупатели: {p1}']].corr(method='spearman')

,Покупатели: Маркет Вконтакте,Покупатели: Партнёр 1
Покупатели: Маркет Вконтакте,1.00,0.93
Покупатели: Партнёр 1,0.93,1.00


In [14]:
eligible = geo[geo[f'Покупатели: {mv}'] >= 1000].copy()
eligible.sort_values('Доля Маркета в сумме покупателей каналов, %', ascending=False)[[f'Покупатели: {mv}', f'Покупатели: {p1}', 'Доля Маркета в сумме покупателей каналов, %']]

,Покупатели: Маркет Вконтакте,Покупатели: Партнёр 1,"Доля Маркета в сумме покупателей каналов, %"
Регион,,,
Новгородская область,"6,021.00",14036,30.02
Челябинская область,"20,465.00",67879,23.17
Курская область,"4,776.00",17171,21.76
Пермский край,"13,366.00",54523,19.69
Владимирская область,"4,754.00",19394,19.69
...,...,...,...
Брянская область,"1,235.00",14161,8.02
Калининградская область,"1,437.00",16855,7.86
Тверская область,"1,738.00",21310,7.54


In [15]:
geo[[f'Заказов на покупателя: {mv}', f'Заказов на покупателя: {p1}']].describe().T

,count,mean,std,min,25%,50%,75%,max
Заказов на покупателя: Маркет Вконтакте,84.00,1.40,0.07,1.26,1.35,1.38,1.43,1.62
Заказов на покупателя: Партнёр 1,88.00,1.57,0.06,1.44,1.53,1.56,1.59,1.76


Москва и Санкт-Петербург — крупнейшие известные регионы Маркета. Суммы покупателей двух каналов не являются уникальными людьми, поэтому показатель доли Маркета в сумме — не проникновение среди населения и не рыночная доля. Фильтр от 1 000 покупателей Маркета уменьшает влияние маленьких чисел, но исключает часть потенциально интересных регионов. Большая доля «Не указан» у партнёра дополнительно ограничивает сравнение географии.

## Концентрация покупателей по регионам

Доля регионов рассчитывается внутри доступных данных. График не показывает охват населения и не доказывает, что региональный сегмент готов к чекауту или уникален по каналу.

In [16]:
regional = pd.DataFrame({
    'Доля Маркета, %': regions[f'Покупатели: {mv}'] / regions[f'Покупатели: {mv}'].sum() * 100,
    'Доля партнёра, %': regions[f'Покупатели: {p1}'] / regions[f'Покупатели: {p1}'].sum() * 100,
}).dropna()
regional['Разница долей Маркет − партнёр, п.п.'] = (
    regional['Доля Маркета, %'] - regional['Доля партнёра, %']
)
top_regions = regional.reindex(regional['Разница долей Маркет − партнёр, п.п.'].abs().sort_values(ascending=False).index).head(12)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=top_regions.index, x=top_regions['Разница долей Маркет − партнёр, п.п.'],
    orientation='h',
    marker=dict(color=np.where(top_regions['Разница долей Маркет − партнёр, п.п.'] >= 0, '#3978C6', '#E07A5F')),
    hovertemplate='%{y}<br>Разница долей: %{x:.2f} п.п.<extra></extra>'
))
fig.update_layout(template='plotly_white', height=560,
                  title='Регионы с наибольшей разницей доли Маркета и партнёра',
                  xaxis_title='Доля Маркета минус доля партнёра, п.п.',
                  yaxis=dict(autorange='reversed'),
                  margin=dict(l=150, r=30, t=80, b=60))
fig.add_vline(x=0, line_width=1, line_color='#777777')
fig.show(config={'displaylogo': False, 'responsive': True})